# Опыт 06: продолжение на 20 000 парах до 50 000 шагов

**Гипотеза:** модель пока недообучена: после 10 000 шагов каждую пару предъявили в среднем четыре раза, validation loss ещё снижается, самостоятельный код слабый. Продолжаем **полученные веса и AdamW с шага 10 000**, меняем только длительность обучения.

В новой Colab-сессии выбери **T4** и выполняй ячейки сверху вниз. При отсутствии PyArrow сначала выполни отдельную ячейку `%pip install pyarrow`. В ячейке загрузки появится выбор файла: загрузи **`hw02-byte-20k-step-10000.pt`** из опыта 05 (переименованная копия с теми же байтами тоже подходит). Checkpoint на 2000 шагов для этого опыта не подходит. Старый results JSON не требуется.

Прежние 20 000 train-пар, 19 960 вопросов, первые 64 train и те же 64 validation; исходные фильтры, split и hashes. Модель 2+2/128, 4 головы, FFN 512, байты, FP32, LR 0.001, AdamW, clipping 1.0, batch 8, dropout 0.1. Веса, optimizer, счётчик и Python/Torch/CUDA RNG восстанавливаются из checkpoint. Версия PyTorch и GPU нового запуска записываются; равенство RNG не гарантирует одинаковые вычисления между разными версиями/устройствами.

Сначала повторная диагностика **10 000** без обновления весов; затем ещё **40 000** обновлений, контроль **20 000 / 30 000 / 50 000 суммарных шагов**. На 50 000 будет 20 предъявлений на пару в среднем, не точное число эпох. По прошлому темпу добавочные шаги — примерно **16 минут чистого обучения плюс диагностики**; это прогноз. Тренировочную ячейку выполнить один раз.

Метрики: прежние loss/AST/`solve`/EOS/длины/exact и новая **`solve_has_value_return`** — `return` со значением в самой функции `solve`, включая ветки, исключая вложенные функции/классы. Программы не исполняются; наличие `return`, AST и exact не являются функциональным качеством. Печатаются три примера каждой части, все 64 сохраняются в JSON.

Таймер обучения охватывает **только добавленные шаги 10 001–50 000**, без loss-check, greedy, печати и записи. Есть отдельный таймер greedy и общий active wall time. Новая AST-проверка `return` входит в greedy timer. Full test/180-секундный замер здесь не выполняются.

Последняя ячейка скачает **один ZIP** с полным JSON, checkpoints 20 000/30 000/50 000 и `latest`, описью SHA-256. В Colab дополнительно выбери **Файл → Скачать → Скачать файл IPYNB**, чтобы сохранить сам notebook с outputs. ZIP не содержит notebook с outputs. Если обучение прервано вручную, можно отдельно выполнить отчёт и скачивание: архив помечает частичный запуск и последний сохранённый checkpoint; он может отставать от последнего логирования.

На подготовке выполнены только короткие CPU-проверки. 40 000 добавочных шагов и T4-запуск агентом не выполнялись.

Источники: [ДЗ 2](https://github.com/ashaba1in/hse-nlp/blob/7f837940c4af9a686fca7bff5f02345218b6fe80/homeworks/hw2_code_generation.md), [датасет](https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/blob/b2724689ff079e9c22efd548fe134bfb29036dbb/README.md), [PyTorch: сохранение и продолжение](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html#saving-loading-a-general-checkpoint-for-inference-and-or-resuming-training), [CUDA RNG](https://docs.pytorch.org/docs/2.11/generated/torch.cuda.set_rng_state_all.html), [Colab upload/download](https://github.com/googlecolab/colabtools/blob/main/google/colab/files.py). Бюджет и порядок проверок — наши решения.


In [1]:
# HW2_BYTE_20K_CONTINUE: resume saved model/AdamW/RNG; only train longer.
import ast
import hashlib
import json
import math
import random
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
import pyarrow.parquet as pq

SEED, TRAIN_SIZE, BATCH_SIZE = 42, 20000, 8
START_STEP = 10000
EVALUATION_STEPS = (20000, 30000, 50000)
TOTAL_STEPS, LOG_EVERY, CHECKPOINT_EVERY = EVALUATION_STEPS[-1], 50, 1000
D_MODEL, N_LAYERS, MAX_POSITIONS = 128, 2, 2048
PAD, BOS, EOS, VOCAB = 0, 1, 2, 259
GENERATION_CAP, INFERENCE_BATCH = 512, 32
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Enable GPU in Colab before running this notebook'
device = torch.device('cuda')
print('Device:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

RESULTS_FILE = Path('hw02-byte-20k-continue-results.json')
LATEST_CHECKPOINT = 'hw02-byte-20k-continue-latest.pt'
def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()
def data_hash(rows):
    return hashlib.sha256(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
def save_results():
    RESULTS_FILE.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')

previous_10000_run = {
    'origin': 'saved user experiment 05 outputs; not measured by this notebook',
    'gpu': 'Tesla T4', 'torch_version': '2.11.0+cu130', 'steps': 10000,
    'validation_loss_16': 1.3694318238007048,
    'train_64_loss': 1.3372655806706149, 'validation_64_loss': 1.3713495915319747,
    'train_64_ast_count': 14, 'validation_64_ast_count': 31,
    'train_64_eos_count': 4, 'validation_64_eos_count': 5,
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 236.7992673299985,
}


Device: Tesla T4
PyTorch: 2.11.0+cu130


In [2]:
# 1. Same first shard/filter/split; expand only the training list.
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
train_file = cache / 'train.parquet'
data_started = time.perf_counter()
if not train_file.exists():
    urlretrieve(f'{base_url}/data/train/part-00000.parquet', train_file)
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
def eligible(row):
    return len(row['question'].encode()) <= 512 and len(row['code'].encode()) < 512

train, train_row_indices, validation = [], [], None
offset, rejected_length, excluded_heldout = 0, 0, 0
for batch in pq.ParquetFile(train_file).iter_batches(batch_size=4096):
    rows = batch.to_pylist()
    if validation is None:
        validation = [r for r in rows if eligible(r) and is_validation(r['question'])][:64]
    for index, row in enumerate(rows):
        if not eligible(row):
            rejected_length += 1
            continue
        if is_validation(row['question']):
            excluded_heldout += 1
            continue
        train.append(row)
        train_row_indices.append(offset + index)
        if len(train) == TRAIN_SIZE:
            break
    if len(train) == TRAIN_SIZE:
        break
    offset += len(rows)
assert len(train) == TRAIN_SIZE and len(validation) == 64
train_hash, validation_hash = data_hash(train), data_hash(validation)
prefix512_hash = data_hash(train[:512])
assert train_hash == '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e'
assert prefix512_hash == '3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35'
assert validation_hash == '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050'
assert all(not is_validation(r['question']) for r in train)
question_counts = Counter(r['question'] for r in train)
assert not (set(question_counts) & {r['question'] for r in validation})
train_diagnostic = train[:64]  # Same 64 train pairs used in the 512-pair experiment.
data_info = {
    'revision': revision, 'shard': 0, 'training_examples': len(train),
    'unique_train_questions': len(question_counts),
    'duplicate_rows_beyond_first_question': len(train) - len(question_counts),
    'max_question_multiplicity': max(question_counts.values()),
    'validation_examples': len(validation),
    'train_sha256': train_hash, 'validation_sha256': validation_hash,
    'train_prefix512_sha256': prefix512_hash,
    'train_row_indices_sha256': data_hash(train_row_indices),
    'first_train_row_index': train_row_indices[0], 'last_train_row_index': train_row_indices[-1],
    'raw_rows_scanned': train_row_indices[-1] + 1,
    'rejected_length_before_last_selected_row': rejected_length,
    'excluded_hash_heldout_before_last_selected_row': excluded_heldout,
    'selection': 'first 20000 eligible non-heldout rows of shard 0; question<=512 bytes, code<512 bytes',
    'question_overlap_train_validation': 0,
    'download_and_selection_seconds': time.perf_counter() - data_started,
}
print('DATA', json.dumps(data_info), flush=True)


DATA {"revision": "b2724689ff079e9c22efd548fe134bfb29036dbb", "shard": 0, "training_examples": 20000, "unique_train_questions": 19960, "duplicate_rows_beyond_first_question": 40, "max_question_multiplicity": 5, "validation_examples": 64, "train_sha256": "1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e", "validation_sha256": "0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050", "train_prefix512_sha256": "3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35", "train_row_indices_sha256": "8280c88a7aa6ba3ebe31bfdf9123e5d3fb7a8b3d57272c6fe7fc7e02abdaf142", "first_train_row_index": 1, "last_train_row_index": 22958, "raw_rows_scanned": 22959, "rejected_length_before_last_selected_row": 691, "excluded_hash_heldout_before_last_selected_row": 2268, "selection": "first 20000 eligible non-heldout rows of shard 0; question<=512 bytes, code<512 bytes", "question_overlap_train_validation": 0, "download_and_selection_seconds": 2.220383804999983}


In [3]:
# 2. Byte encoding: reversible, no tokenizer training or external vocabulary.
def encode(text):
    return [b + 3 for b in text.encode('utf-8')]
def decode(ids):
    return bytes(i - 3 for i in ids if 3 <= i < VOCAB).decode('utf-8', errors='replace')
def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS]) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c) for c in codes], batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS]) for c in codes], batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels



In [4]:
# 3. Standard encoder-decoder: 2 + 2 layers, learned positions, ReLU, Post-LN.
class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print('Parameters:', sum(p.numel() for p in model.parameters()))



Parameters: 1254915


In [5]:
# 4. Load the exact supplied checkpoint: weights, AdamW, counter, RNG.
RESUME_CHECKPOINT = Path('hw02-byte-20k-step-10000.pt')
SOURCE_SHA256 = '4ed8ff448e62e58ae654b6af9d78a78af2e5309561674592a35511ef17ce001a'
if not RESUME_CHECKPOINT.is_file():
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload one checkpoint from step 10000'
    RESUME_CHECKPOINT = Path(next(iter(uploaded)))
    del uploaded

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def restore_checkpoint(path, expected_sha256):
    assert file_hash(path) == expected_sha256, 'This is not the supplied experiment 05 checkpoint at step 10000'
    state = torch.load(path, map_location='cpu', weights_only=True)
    expected_config = {'seed': SEED, 'batch_size_train': BATCH_SIZE, 'total_steps': START_STEP,
        'evaluation_steps': [2000, 5000, 10000], 'd_model': D_MODEL,
        'encoder_layers': N_LAYERS, 'decoder_layers': N_LAYERS, 'heads': 4,
        'ffn_dim': 4 * D_MODEL, 'dropout': 0.1, 'norm_first': False, 'lr': 0.001,
        'optimizer': 'AdamW', 'weight_decay': 0.01, 'clip_norm': 1.0, 'scheduler': None,
        'tokenizer': 'UTF-8 bytes', 'max_positions': MAX_POSITIONS,
        'generation_cap': GENERATION_CAP, 'batch_size_inference': INFERENCE_BATCH}
    assert state['steps'] == START_STEP and state['config'] == expected_config, 'Checkpoint configuration mismatch'
    assert state['data_revision'] == revision, 'Data revision mismatch'
    assert state['train_sha256'] == train_hash and state['validation_sha256'] == validation_hash, 'Data hash mismatch'
    assert {int(s['step']) for s in state['optimizer']['state'].values()} == {START_STEP}, 'AdamW counter mismatch'
    if device.type == 'cuda':
        assert len(state['cuda_rng_state_all']) == torch.cuda.device_count(), 'GPU count differs from checkpoint'
    model.load_state_dict(state['model'], strict=True)
    optimizer.load_state_dict(state['optimizer'])
    assert {int(s['step']) for s in optimizer.state.values()} == {START_STEP}
    assert next(model.parameters()).dtype == torch.float32
    random.setstate(state['python_random_state'])
    torch.set_rng_state(state['torch_rng_state'])
    if device.type == 'cuda':
        torch.cuda.set_rng_state_all(state['cuda_rng_state_all'])
    return {'source_filename': Path(path).name, 'source_sha256': expected_sha256,
        'source_steps': state['steps'], 'source_config': state['config'],
        'weights_optimizer_python_and_cpu_rng_restored': True,
        'cuda_rng_restored': device.type == 'cuda',
        'source_torch_version_from_saved_notebook': previous_10000_run['torch_version'],
        'current_torch_version': torch.__version__,
        'torch_version_matches_source_notebook': torch.__version__ == previous_10000_run['torch_version']}

resume_info = restore_checkpoint(RESUME_CHECKPOINT, SOURCE_SHA256)
config = {**resume_info['source_config'], 'total_steps': TOTAL_STEPS,
          'evaluation_steps': list(EVALUATION_STEPS)}
print('RESUME', json.dumps(resume_info), flush=True)


Saving hw02-byte-20k-step-10000.pt to hw02-byte-20k-step-10000.pt
RESUME {"source_filename": "hw02-byte-20k-step-10000.pt", "source_sha256": "4ed8ff448e62e58ae654b6af9d78a78af2e5309561674592a35511ef17ce001a", "source_steps": 10000, "source_config": {"seed": 42, "batch_size_train": 8, "total_steps": 10000, "evaluation_steps": [2000, 5000, 10000], "d_model": 128, "encoder_layers": 2, "decoder_layers": 2, "heads": 4, "ffn_dim": 512, "dropout": 0.1, "norm_first": false, "lr": 0.001, "optimizer": "AdamW", "weight_decay": 0.01, "clip_norm": 1.0, "scheduler": null, "tokenizer": "UTF-8 bytes", "max_positions": 2048, "generation_cap": 512, "batch_size_inference": 32}, "weights_optimizer_python_and_cpu_rng_restored": true, "cuda_rng_restored": true, "source_torch_version_from_saved_notebook": "2.11.0+cu130", "current_torch_version": "2.11.0+cu130", "torch_version_matches_source_notebook": true}


In [6]:
# 5. Same diagnostics plus a value-return inside solve (never execute code).
@torch.inference_mode()
def evaluate_loss(rows):
    model.eval()
    loss_sum, label_count = 0.0, 0
    for offset in range(0, len(rows), 16):
        src, tgt, labels = training_batch(rows[offset:offset + 16])
        logits = model(src, tgt)
        loss_sum += nn.functional.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
                                               ignore_index=PAD, reduction='sum').item()
        label_count += labels.ne(PAD).sum().item()
    return loss_sum / label_count

def solve_value_return(tree):
    # A nested function's return does not return a value from solve itself.
    def contains_value_return(node):
        if isinstance(node, ast.Return):
            return node.value is not None and not (isinstance(node.value, ast.Constant) and node.value.value is None)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):
            return False
        return any(contains_value_return(child) for child in ast.iter_child_nodes(node))
    return any(isinstance(node, ast.FunctionDef) and node.name == 'solve'
               and any(contains_value_return(stmt) for stmt in node.body) for node in tree.body)

def inspect_ids(row, ids):
    active = [i for i in ids if i != PAD]
    code = decode(active)
    try:
        tree = ast.parse(code)
        parses = True
        defines_solve = any(isinstance(node, ast.FunctionDef) and node.name == 'solve'
                            for node in tree.body)
        value_return = solve_value_return(tree)
    except (SyntaxError, ValueError):
        parses, defines_solve, value_return = False, False, False
    return {'question': row['question'], 'reference_code': row['code'], 'generated_code': code,
        'generated_tokens': len(active), 'expected_tokens_including_eos': len(encode(row['code'])) + 1,
        'eos_emitted': EOS in active,
        'early_eos': EOS in active and active.index(EOS) + 1 < GENERATION_CAP,
        'reached_cap': len(active) == GENERATION_CAP,
        'parses_as_python': parses, 'defines_solve': defines_solve,
        'solve_has_value_return': value_return,
        'exact_code_match': code == row['code'],
        'exact_tokens_with_eos': active == encode(row['code']) + [EOS]}

@torch.inference_mode()
def inspect_split(name, rows):
    model.eval()
    fixed_loss = evaluate_loss(rows)  # Excluded from the generation timer.
    sync()
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    generation_started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        generated = model.generate(source_batch([r['question'] for r in batch]),
                                   max_new_tokens=GENERATION_CAP).cpu().tolist()
        examples.extend(inspect_ids(row, ids) for row, ids in zip(batch, generated))
    sync()
    seconds = time.perf_counter() - generation_started
    lengths = [e['generated_tokens'] for e in examples]
    summary = {'split': name, 'examples': len(rows), 'loss': fixed_loss,
        'mean_generated_tokens': sum(lengths) / len(lengths),
        'min_generated_tokens': min(lengths), 'max_generated_tokens': max(lengths),
        'seconds': seconds, 'answers_per_second': len(rows) / seconds,
        'generated_tokens': sum(lengths), 'tokens_per_second': sum(lengths) / seconds,
        'batch_size_inference': INFERENCE_BATCH, 'max_new_tokens': GENERATION_CAP,
        'timer_scope': 'encoding, transfers, greedy, CPU decoding and AST/return/exact checks; loss/checkpoint excluded',
        'peak_gpu_memory_mib': torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None}
    for metric in ['eos_emitted', 'early_eos', 'reached_cap', 'parses_as_python', 'defines_solve',
                   'solve_has_value_return', 'exact_code_match', 'exact_tokens_with_eos']:
        summary[metric + '_count'] = sum(e[metric] for e in examples)
        summary[metric + '_fraction'] = summary[metric + '_count'] / len(examples)
    print('DIAGNOSTICS', json.dumps(summary), flush=True)
    for example in examples[:3]:
        print('EXAMPLE', name, json.dumps(example, ensure_ascii=False), flush=True)
    return {'summary': summary, 'examples': examples}

def save_checkpoint(step):
    state = {'model': {k: v.cpu() for k, v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(), 'steps': step, 'config': config,
        'data_revision': revision, 'train_sha256': train_hash, 'validation_sha256': validation_hash,
        'python_random_state': random.getstate(), 'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}
    temporary = Path(LATEST_CHECKPOINT).with_suffix('.tmp')
    torch.save(state, temporary)
    temporary.replace(LATEST_CHECKPOINT)
    result['latest_checkpoint_step'] = step
    if step in EVALUATION_STEPS:
        name = f'hw02-byte-20k-continue-step-{step}.pt'
        torch.save(state, name)
        result['checkpoint_files'][str(step)] = name


In [7]:
# 6. Continue from 10001; only the added updates enter the training timer.
assert not globals().get('continuation_started', False), 'Training cell already started; use a fresh runtime to restore'
assert {int(s['step']) for s in optimizer.state.values()} == {START_STEP}, 'Restore the checkpoint before training'
assert resume_info['source_steps'] == START_STEP
continuation_started = True
result = {'experiment': '06-byte-20k-continue-10000-to-50000', 'config': config, 'data': data_info,
    'resume': resume_info, 'previous_10000_run': previous_10000_run,
    'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU local check',
    'torch_version': torch.__version__, 'dtype': str(next(model.parameters()).dtype),
    'parameters': sum(p.numel() for p in model.parameters()), 'start_step': START_STEP,
    'actual_steps': START_STEP, 'updates_completed_this_run': 0,
    'training_log': [], 'diagnostics': {}, 'checkpoint_files': {},
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0,
    'active_wall_seconds_including_diagnostics_and_checkpointing': 0.0,
    'training_timer_scope': 'only updates after start_step; batch construction, transfers, forward/backward, clipping, AdamW, block-end CUDA sync',
    'run_finished': False, 'functional_quality_evaluated': False, 'test_evaluated': False,
    'generalization_loss_and_code_diagnostics_evaluated': False}
sync()
run_started = time.perf_counter()
save_checkpoint(START_STEP)
save_results()
initial = {'step': START_STEP, 'train_batch_loss': None, 'validation_loss_16': evaluate_loss(validation[:16]),
    'updates_completed_this_run': 0, 'mean_training_pair_presentations': START_STEP * BATCH_SIZE / len(train),
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0}
result['training_log'].append(initial)
print('LOSS', json.dumps(initial), flush=True)
save_results()

def run_diagnostics(step):
    model.generate(source_batch([validation[0]['question']]), max_new_tokens=8)
    result['diagnostics'][str(step)] = {}
    for name, fixed_rows in [('train_64', train_diagnostic), ('validation_64', validation)]:
        result['diagnostics'][str(step)][name] = inspect_split(name, fixed_rows)
        save_results()
    result['generalization_loss_and_code_diagnostics_evaluated'] = True

run_diagnostics(START_STEP)  # Fresh measurement at loaded weights, without an update.
save_results()
sync()
block_started = time.perf_counter()
training_seconds = 0.0

for step in range(START_STEP + 1, TOTAL_STEPS + 1):
    model.train()
    src, tgt, labels = training_batch(random.sample(train, BATCH_SIZE))
    logits = model(src, tgt)
    loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if (step == START_STEP + 1 or step % LOG_EVERY == 0 or step % CHECKPOINT_EVERY == 0
            or step in EVALUATION_STEPS or step == TOTAL_STEPS):
        sync()
        training_seconds += time.perf_counter() - block_started
        result['actual_steps'] = step
        result['updates_completed_this_run'] = step - START_STEP
        result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
        entry = {'step': step, 'train_batch_loss': loss.item(),
            'validation_loss_16': evaluate_loss(validation[:16]), 'updates_completed_this_run': step - START_STEP,
            'training_seconds_excluding_evaluation_logging_and_checkpoints': training_seconds,
            'training_updates_per_second_excluding_evaluation_logging_and_checkpoints': (step - START_STEP) / training_seconds,
            'mean_training_pair_presentations': step * BATCH_SIZE / len(train)}
        result['training_log'].append(entry)
        print('LOSS', json.dumps(entry), flush=True)
        result['active_wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - run_started
        save_results()
        if step in EVALUATION_STEPS:
            run_diagnostics(step)
        if step % CHECKPOINT_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
            save_checkpoint(step)
            print('CHECKPOINT', step, LATEST_CHECKPOINT, flush=True)
        result['active_wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - run_started
        save_results()
        sync()
        block_started = time.perf_counter()

sync()
result['run_finished'] = True
result['active_wall_seconds_including_diagnostics_and_checkpointing'] = time.perf_counter() - run_started
save_results()


LOSS {"step": 10000, "train_batch_loss": null, "validation_loss_16": 1.3694318238007048, "updates_completed_this_run": 0, "mean_training_pair_presentations": 4.0, "training_seconds_excluding_evaluation_logging_and_checkpoints": 0.0}
DIAGNOSTICS {"split": "train_64", "examples": 64, "loss": 1.3372655806706149, "mean_generated_tokens": 491.5625, "min_generated_tokens": 154, "max_generated_tokens": 512, "seconds": 12.142844521999962, "answers_per_second": 5.270593713363219, "generated_tokens": 31460, "tokens_per_second": 2590.8262222251074, "batch_size_inference": 32, "max_new_tokens": 512, "timer_scope": "encoding, transfers, greedy, CPU decoding and AST/return/exact checks; loss/checkpoint excluded", "peak_gpu_memory_mib": 497.05126953125, "eos_emitted_count": 4, "eos_emitted_fraction": 0.0625, "early_eos_count": 4, "early_eos_fraction": 0.0625, "reached_cap_count": 60, "reached_cap_fraction": 0.9375, "parses_as_python_count": 14, "parses_as_python_fraction": 0.21875, "defines_solve_cou

In [8]:
# 7. Available checkpoints; this also works after a manual interruption.
for point in (START_STEP,) + EVALUATION_STEPS:
    checked = result['diagnostics'].get(str(point), {})
    if not all(name in checked for name in ('train_64', 'validation_64')):
        continue
    loss_entry = next(e for e in result['training_log'] if e['step'] == point)
    print('SUMMARY', json.dumps({'step': point, 'updates_completed_this_run': point - START_STEP,
        'validation_loss_16': loss_entry['validation_loss_16'],
        'mean_training_pair_presentations': point * BATCH_SIZE / len(train),
        'training_seconds_excluding_evaluation_logging_and_checkpoints':
            loss_entry['training_seconds_excluding_evaluation_logging_and_checkpoints'],
        'train_64': checked['train_64']['summary'], 'validation_64': checked['validation_64']['summary']}), flush=True)
print('RESULT', json.dumps({k: result[k] for k in ['start_step', 'actual_steps', 'updates_completed_this_run',
    'run_finished', 'training_seconds_excluding_evaluation_logging_and_checkpoints',
    'active_wall_seconds_including_diagnostics_and_checkpointing', 'latest_checkpoint_step', 'checkpoint_files']}), flush=True)
print('Download ZIP in the next cell, and save the notebook with outputs separately.')


SUMMARY {"step": 10000, "updates_completed_this_run": 0, "validation_loss_16": 1.3694318238007048, "mean_training_pair_presentations": 4.0, "training_seconds_excluding_evaluation_logging_and_checkpoints": 0.0, "train_64": {"split": "train_64", "examples": 64, "loss": 1.3372655806706149, "mean_generated_tokens": 491.5625, "min_generated_tokens": 154, "max_generated_tokens": 512, "seconds": 12.142844521999962, "answers_per_second": 5.270593713363219, "generated_tokens": 31460, "tokens_per_second": 2590.8262222251074, "batch_size_inference": 32, "max_new_tokens": 512, "timer_scope": "encoding, transfers, greedy, CPU decoding and AST/return/exact checks; loss/checkpoint excluded", "peak_gpu_memory_mib": 497.05126953125, "eos_emitted_count": 4, "eos_emitted_fraction": 0.0625, "early_eos_count": 4, "early_eos_fraction": 0.0625, "reached_cap_count": 60, "reached_cap_fraction": 0.9375, "parses_as_python_count": 14, "parses_as_python_fraction": 0.21875, "defines_solve_count": 14, "defines_solve

In [9]:
# 8. Download one archive: full JSON, all new control checkpoints, and latest.
def build_results_archive():
    save_results()  # Include current metadata even if training was interrupted.
    members = [RESULTS_FILE, Path(LATEST_CHECKPOINT)] + [Path(p) for p in result['checkpoint_files'].values()]
    assert all(p.is_file() for p in members), 'A listed results/checkpoint file is missing'
    manifest = {'experiment': result['experiment'], 'start_step': result['start_step'],
        'actual_steps': result['actual_steps'], 'run_finished': result['run_finished'],
        'latest_checkpoint_step': result['latest_checkpoint_step'],
        'source_checkpoint_sha256': resume_info['source_sha256'],
        'notebook_with_outputs_included': False,
        'files': [{'filename': p.name, 'bytes': p.stat().st_size, 'sha256': file_hash(p)} for p in members]}
    archive_path = Path(f'hw02-byte-20k-continue-{START_STEP}-to-{TOTAL_STEPS}.zip')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for p in members:
            bundle.write(p, arcname=p.name)
        bundle.writestr('manifest.json', json.dumps(manifest, ensure_ascii=False, indent=2))
        bundle.writestr('README.txt', 'Save the Colab notebook with outputs separately using File > Download > Download .ipynb.\n'
            'AST, value-return and exact metrics are diagnostics, not functional correctness.\n'
            'run_finished may be false; latest_checkpoint_step can be earlier than actual_steps.\n')
    print('ARCHIVE', archive_path.name, 'bytes', archive_path.stat().st_size,
          'finished', result['run_finished'], 'latest checkpoint', result['latest_checkpoint_step'])
    return archive_path

archive_path = build_results_archive()
from google.colab import files
files.download(str(archive_path))


ARCHIVE hw02-byte-20k-continue-10000-to-50000.zip bytes 49526861 finished True latest checkpoint 50000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>